# Préparation du jeu de données CelebA (local) — VERSION CORRIGÉE

**À exécuter en local uniquement.** Toutes les images source sont dans un dossier nommé **`DATASET`** à la racine du projet.

Ce notebook réorganise les données depuis **`DATASET/CelebA/`** vers **`data/processed/`** :
- **data/processed/train/metadata.csv** : métadonnées d'entraînement (~182 339 images)
- **data/processed/val/metadata.csv** : métadonnées de validation (~20 260 images)
- **data/processed/statistics.json** : statistiques du dataset

⚠️  **STRUCTURE REQUISE:**
```
DATASET/
├── CelebA/
│   ├── img_align_celeba/           (202 599 images)
│   └── metadata/
│       ├── list_attr_celeba.txt    (40 attributs)
│       └── identity_CelebA.txt     (ID personne)
```

Les images restent dans **DATASET/CelebA/** (pas de copie).  
Les métadonnées CSV contiennent les chemins relatifs vers les images.  
Seed: **42** pour reproductibilité.

---
### ✅ Corrections appliquées dans cette version
| # | Problème original | Correction |
|---|---|---|
| 1 | **Bug grammaire prompts** : `"un portrait d'un jeune une femme"` (article masculin + syntagme féminin) | `"un portrait d'une jeune femme"` — accord correct pour les 4 combinaisons genre×âge |
| 2 | **Colonnes manquantes** : `domain` et `class_name` absents du CSV exporté | Exportées dans le CSV (requises par `Text2ImageDataset` et le weighted sampling) |
| 3 | **Split aléatoire simple** | **Split stratifié** 90/10 par genre×âge (distribution équilibrée garantie en validation) |
| 4 | **statistics.json minimal** | Enrichi avec `classes`, `domains`, `prompt_stats`, `quality_issues` (requis par `visualize.py`) |

## 1. Configuration des chemins et constantes

**Objectif** : Configurer les chemins d'accès et les paramètres de base pour la préparation.

In [ ]:
from pathlib import Path
import json
import random
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Remonte l'arborescence jusqu'à trouver le dossier DATASET/
def find_project_root():
    current = Path.cwd().resolve()
    for _ in range(5):
        if (current / "DATASET").exists():
            return current
        parent = current.parent
        if parent == current:
            break
        current = parent
    return Path.cwd().resolve()

PROJECT_ROOT   = find_project_root()
DATASET_ROOT   = PROJECT_ROOT / "DATASET"
DATA_ROOT      = PROJECT_ROOT / "data"
PROCESSED_ROOT = DATA_ROOT / "processed"

# Créer data/ (métadonnées seulement, pas d'images)
DATA_ROOT.mkdir(parents=True, exist_ok=True)
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)
(PROCESSED_ROOT / "train").mkdir(parents=True, exist_ok=True)
(PROCESSED_ROOT / "val").mkdir(parents=True, exist_ok=True)

MAX_PER_DOMAIN = None  # None = toutes les images (202 599)

print(f'PROJECT_ROOT  : {PROJECT_ROOT}')
print(f'DATASET_ROOT  : {DATASET_ROOT}')
print(f'PROCESSED_ROOT: {PROCESSED_ROOT}')
print(f'\n✓ Répertoire DATASET existe : {DATASET_ROOT.exists()}')
print(f'✓ Répertoire data/ créé     : {DATA_ROOT.exists()}')

## 2. Générateur de prompts — CORRIGÉ

**Objectif** : Construire des prompts français grammaticalement corrects à partir des 40 attributs CelebA.

**Bug corrigé :**
- ❌ `"un portrait d'un jeune une femme avec cheveux bruns"` — article masculin `d'un` + syntagme féminin `une femme`
- ✅ `"un portrait d'une jeune femme avec cheveux bruns"` — accord correct

**Les 4 combinaisons genre × âge :**

| Male | Young | Prompt généré |
|------|-------|---------------|
| 1 | 1 | `un portrait d'un jeune homme ...` |
| 1 | 0 | `un portrait d'un homme plus âgé ...` |
| 0 | 1 | `un portrait d'une jeune femme ...` |
| 0 | 0 | `un portrait d'une femme plus âgée ...` |

In [ ]:
def build_prompt(attrs: dict) -> str:
    """
    Construit un prompt français grammaticalement correct à partir des
    attributs CelebA.

    CORRECTION DU BUG ORIGINAL :
      Ancien code :
        gender = 'un homme' if Male else 'une femme'
        age    = 'jeune' if Young else 'plus âgé'
        prompt = f"un portrait d'un {age} {gender} ..."
      → Produisait : "un portrait d'un jeune une femme" (incohérent)

      Correction : construire le syntagme en un seul bloc :
      → Produit   : "un portrait d'une jeune femme" (correct)
    """
    is_male  = attrs.get('Male',  0) == 1
    is_young = attrs.get('Young', 0) == 1

    # Syntagme 'de + article + sujet' en un seul bloc
    # (évite la duplication d'article ex: 'd'un un jeune homme')
    if is_male and is_young:
        subject_phrase = "d'un jeune homme"
    elif is_male and not is_young:
        subject_phrase = "d'un homme plus âgé"
    elif not is_male and is_young:
        subject_phrase = "d'une jeune femme"
    else:
        subject_phrase = "d'une femme plus âgée"

    features = []

    # Couleur de cheveux (priorité : noir > brun > blond > gris > chauve)
    if attrs.get('Black_Hair',  0):
        features.append('cheveux noirs')
    elif attrs.get('Brown_Hair', 0):
        features.append('cheveux bruns')
    elif attrs.get('Blond_Hair', 0):
        features.append('cheveux blonds')
    elif attrs.get('Gray_Hair',  0):
        features.append('cheveux gris')
    elif attrs.get('Bald',       0):
        features.append('chauve')

    # Expression faciale (accord en genre)
    if attrs.get('Smiling', 0):
        features.append('souriant' if is_male else 'souriante')

    # Accessoires
    if attrs.get('Eyeglasses',       0):
        features.append('lunettes')
    if attrs.get('Wearing_Hat',      0):
        features.append('chapeau')
    if attrs.get('Wearing_Earrings', 0):
        features.append("boucles d'oreilles")

    if features:
        return f"un portrait {subject_phrase} avec {', '.join(features)}"
    return f"un portrait {subject_phrase}"


# Vérification des 4 combinaisons genre × âge
test_cases = [
    ({'Male': 1, 'Young': 1, 'Brown_Hair': 1, 'Smiling': 1}, 'homme jeune souriant'),
    ({'Male': 1, 'Young': 0, 'Black_Hair': 1},               'homme âgé'),
    ({'Male': 0, 'Young': 1, 'Blond_Hair': 1, 'Wearing_Earrings': 1}, 'femme jeune'),
    ({'Male': 0, 'Young': 0, 'Gray_Hair': 1},                'femme âgée'),
    ({'Male': 0, 'Young': 1},                                'femme sans features'),
]
print('Vérification des prompts corrigés :')
for attrs, label in test_cases:
    p = build_prompt(attrs)
    print(f'  [{label}] → {p}')

## 3. Préparation CelebA (202 599 images de visages)

**Objectif** : Charger les 40 attributs depuis `list_attr_celeba.txt`, générer les prompts corrigés et créer les lignes du dataset.

In [ ]:
# Les 40 noms d'attributs CelebA (ordre exact du fichier list_attr_celeba.txt)
ATTR_NAMES = [
    '5_o_Clock_Shadow', 'Arched_Eyebrows', 'Attractive', 'Bags_Under_Eyes',
    'Bald', 'Bangs', 'Big_Lips', 'Big_Nose', 'Black_Hair', 'Blond_Hair',
    'Blurry', 'Brown_Hair', 'Bushy_Eyebrows', 'Chubby', 'Double_Chin',
    'Eyeglasses', 'Goatee', 'Gray_Hair', 'Heavy_Makeup', 'High_Cheekbones',
    'Male', 'Mouth_Slightly_Open', 'Mustache', 'Narrow_Eyes', 'No_Beard',
    'Oval_Face', 'Pale_Skin', 'Pointy_Nose', 'Receding_Hairline', 'Rosy_Cheeks',
    'Sideburns', 'Smiling', 'Straight_Hair', 'Wavy_Hair', 'Wearing_Earrings',
    'Wearing_Hat', 'Wearing_Lipstick', 'Wearing_Necklace', 'Wearing_Necktie', 'Young',
]


def prepare_celeba():
    """
    Charge les attributs CelebA et génère les lignes du dataset.
    Retourne une liste de dicts : image_path, prompt, domain, class_name.

    Notes :
      - domain et class_name sont inclus (requis par Text2ImageDataset)
      - les prompts sont générés avec build_prompt() corrigé
    """
    img_dir      = DATASET_ROOT / 'CelebA' / 'img_align_celeba'
    metadata_dir = DATASET_ROOT / 'CelebA' / 'metadata'
    attr_path    = metadata_dir / 'list_attr_celeba.txt'

    if not attr_path.exists():
        raise FileNotFoundError(f'Fichier non trouvé: {attr_path}')
    if not img_dir.exists():
        raise FileNotFoundError(f'Répertoire non trouvé: {img_dir}')

    print(f'Chargement CelebA depuis {img_dir}')

    # Charger les attributs (ligne 1 = count, ligne 2 = noms, suite = données)
    attributes = {}
    with open(attr_path, 'r') as f:
        lines = f.readlines()

    for line in lines[2:]:
        parts = line.strip().split()
        if len(parts) >= 41:
            img_name    = parts[0]
            attr_values = [int(v) for v in parts[1:41]]
            # Convertir -1/+1 → 0/1
            attributes[img_name] = {
                ATTR_NAMES[i]: (1 if v > 0 else 0)
                for i, v in enumerate(attr_values)
            }

    print(f'Attributs chargés : {len(attributes)} images')

    # Générer les lignes du dataset
    rows  = []
    count = 0

    for img_path in sorted(img_dir.glob('*.jpg')):
        if MAX_PER_DOMAIN and count >= MAX_PER_DOMAIN:
            break
        img_file = img_path.name
        if img_file not in attributes:
            continue

        attrs  = attributes[img_file]
        prompt = build_prompt(attrs)  # prompts corrigés

        rows.append({
            'image_path': f'CelebA/img_align_celeba/{img_file}',
            'prompt':     prompt,
            'domain':     'celeba',         # requis par Text2ImageDataset
            'class_name': 'visage_humain',   # requis par weighted sampling
        })
        count += 1

    print(f'Prompts générés : {len(rows)} images')
    return rows


print('\n' + '='*60)
print('Préparation CelebA (202 599 images)...')
print('='*60)
celeba_rows = prepare_celeba()
print(f'  → {len(celeba_rows)} images')

print('\nExemples de prompts corrigés :')
for row in celeba_rows[:5]:
    print(f'  • {row["prompt"]}')

## 4. Split stratifié 90/10 et sauvegarde

**Objectif** : Diviser le dataset en train/val avec un split stratifié par genre × âge, puis sauvegarder les métadonnées CSV et le fichier de statistiques.

**Pourquoi stratifié ?**  
Le split aléatoire simple peut créer un déséquilibre dans le jeu de validation (ex : trop peu de femmes âgées si le tirage est défavorable).  
Le split stratifié garantit que chaque strate (`jeune_homme`, `homme_âgé`, `jeune_femme`, `femme_âgée`) est représentée proportionnellement dans les deux splits.

In [ ]:
print('\n' + '='*60)
print('ÉCRITURE DES MÉTADONNÉES')
print('='*60)

# Créer le DataFrame
df = pd.DataFrame(celeba_rows)

print(f'\nDataFrame CelebA : {df.shape}')
print(f'Colonnes : {list(df.columns)}')
print(f'\nPremières lignes :')
print(df.head(3).to_string())


# ── Split stratifié 90/10 par genre × âge ────────────────────────────
def get_strate(prompt: str) -> str:
    """Extrait la strate genre × âge depuis le prompt corrigé."""
    if 'jeune homme'       in prompt: return 'jeune_homme'
    elif 'homme plus âgé'  in prompt: return 'homme_age'
    elif 'jeune femme'     in prompt: return 'jeune_femme'
    else:                             return 'femme_agee'

df['strate'] = df['prompt'].apply(get_strate)

print(f'\nDistribution des strates :')
for strate, cnt in df['strate'].value_counts().items():
    print(f'  {strate:<20} : {cnt:>7} images ({cnt/len(df)*100:.1f}%)')

# Split par strate (garantit distribution équilibrée dans val)
np.random.seed(SEED)
train_indices, val_indices = [], []

for strate, group in df.groupby('strate'):
    idx = group.index.tolist()
    np.random.shuffle(idx)
    split = int(len(idx) * 0.9)
    train_indices.extend(idx[:split])
    val_indices.extend(idx[split:])

train_indices = sorted(train_indices)
val_indices   = sorted(val_indices)

df_train = df.loc[train_indices]
df_val   = df.loc[val_indices]

print(f'\nSplit stratifié 90/10 :')
print(f'  Train : {len(df_train)} images (90%)')
print(f'  Val   : {len(df_val)}   images (10%)')

print(f'\nDistribution strates dans Val :')
for strate, cnt in df_val['strate'].value_counts().items():
    print(f'  {strate:<20} : {cnt:>6} ({cnt/len(df_val)*100:.1f}%)')


# ── Sauvegarde metadata.csv (avec domain + class_name) ───────────────
# domain et class_name sont requis par Text2ImageDataset (weighted sampling)
COLUMNS_CSV = ['image_path', 'prompt', 'domain', 'class_name']

train_csv = PROCESSED_ROOT / 'train' / 'metadata.csv'
val_csv   = PROCESSED_ROOT / 'val'   / 'metadata.csv'

df_train[COLUMNS_CSV].to_csv(train_csv, index=False)
df_val[COLUMNS_CSV].to_csv(val_csv,   index=False)

print(f'\n✅ Métadonnées train sauvegardées : {train_csv}')
print(f'   {len(df_train)} images')
print(f'\n✅ Métadonnées val sauvegardées : {val_csv}')
print(f'   {len(df_val)} images')

# Vérification rapide
df_check = pd.read_csv(train_csv)
print(f'\nVérification colonnes exportées : {list(df_check.columns)}')
print(f'Premier prompt : {df_check["prompt"].iloc[0]}')


# ── Sauvegarde statistics.json (enrichi) ─────────────────────────────
# Requis par visualize.plot_dataset_distribution et load_sampling_weights
stats = {
    'total_images':         len(df),
    'train_images':         len(df_train),
    'val_images':           len(df_val),
    'split_ratio':          '90/10 stratifié (genre x âge)',
    'total_attributes':     40,
    'unique_domains':       int(df['domain'].nunique()),
    'classes':              {'visage_humain': len(df)},
    'domains':              {'celeba': len(df)},
    'domain_balance_ratio': 1.0,
    'class_balance_ratio':  1.0,
    'strate_distribution':  df['strate'].value_counts().to_dict(),
    'quality_issues':       {'corrupted': [], 'missing': []},
    'prompt_stats': {
        'unique_prompts':    int(df['prompt'].nunique()),
        'avg_prompt_length': round(df['prompt'].str.split().str.len().mean(), 1),
    },
}

stats_path = PROCESSED_ROOT / 'statistics.json'
with open(stats_path, 'w', encoding='utf-8') as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)

print(f'\n✅ Statistiques sauvegardées : {stats_path}')
print(json.dumps(
    {k: v for k, v in stats.items()
     if k not in ('strate_distribution', 'quality_issues', 'prompt_stats')},
    indent=2, ensure_ascii=False
))

print(f'\n' + '='*60)
print('✅ PRÉPARATION DU DATASET CELEBA COMPLÈTE !')
print('='*60)
print(f'\nFichiers créés :')
print(f'  1. data/processed/train/metadata.csv  ({len(df_train)} images)')
print(f'  2. data/processed/val/metadata.csv    ({len(df_val)} images)')
print(f'  3. data/processed/statistics.json')
print(f'\n💡 Prêt à uploader sur Kaggle !')